Project
DATA.ML.360 Recommeder Systems
Autumn 2024

Selina Rautakoski, selina.rautakoski@tuni.fi (439869)

In [1]:
import pandas as pd
import numpy as np
from scipy import spatial
from scipy.sparse import csr_matrix
from scipy.spatial.distance import pdist, squareform
import math

PART I

(a) Download the MovieLens 100K rating dataset from
https://grouplens.org/datasets/movielens/ (the small dataset recommended for
education and development). Read the dataset, display the first few rows to understand
it, and display the count of ratings (rows) in the dataset to be sure that you download it
correctly.

In [2]:
# Read the dataset from files
moviesData = pd.read_csv("./ml-latest-small/movies.csv")
linksData = pd.read_csv("./ml-latest-small/links.csv")
ratingsData = pd.read_csv("./ml-latest-small/ratings.csv")
tagsData = pd.read_csv("./ml-latest-small/tags.csv")

In [3]:
# First rows of movies
moviesData.head(3)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


In [4]:
# the number of the ratings
ratingsData.shape[0]

100836

In [5]:
ratingsData.head(4)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815


In [6]:
# load data before to make processes more efficient
users = ratingsData['userId'].unique()
items = ratingsData['movieId'].unique()
user_map = {user: i for i, user in enumerate(users)}
item_map = {item: i for i, item in enumerate(items)}
rows = ratingsData['userId'].map(user_map).values
cols = ratingsData['movieId'].map(item_map).values
data = ratingsData['rating'].values
user_item_matrix = csr_matrix((data, (rows, cols)), shape=(len(users), len(items))) # user-item matrix

(b) Implement the user-based collaborative filtering approach, using the Pearson
correlation function for computing similarities between users (4 points)

In [7]:
def similarity(a,b):

    a_ratings = user_item_matrix.getrow(a).toarray().flatten() # all the ratings of a
    a_rating_avg = a_ratings.mean() # average rating of the target user

    b_ratings = user_item_matrix.getrow(b).toarray().flatten() # all the ratings of b
    b_rating_avg = b_ratings.mean() # average rating of b

    common_ratings = (a_ratings > 0) & (b_ratings > 0) #common ratings
    if(len(common_ratings) == 0):  # no common ratings, no similarity
        return 0
    
    a_common_ratings = a_ratings[common_ratings] # ratings of a in common ratings
    b_common_ratings = b_ratings[common_ratings] # ratings of b in common ratings

    a_normalized = a_common_ratings - a_rating_avg # normalized by the average
    b_normalized = b_common_ratings - b_rating_avg

    numerator = np.dot(a_normalized, b_normalized)
    denominator = np.sqrt(np.sum(a_normalized**2)) * np.sqrt(np.sum(b_normalized**2))

    if denominator == 0: # denominator cannot be 0
        return 0
    
    return numerator/denominator # return  the similarity

(c) the
prediction function presented in class for predicting movies scores (4 points).

In [8]:
def prediction(a, movie):

    if a not in user_map or movie not in item_map: # check that the user id and movie id are valid
        return 0
    
    a_idx = user_map[a] # index of a
    movie_idx = item_map[movie] #index of movie

    a_ratings = user_item_matrix[a_idx].toarray().flatten() # all the ratings of a
    a_rating_avg = a_ratings[a_ratings>0].mean() if (len(a_ratings)>0) else 0 # average rating of a, if no ratings this is 0

    sum_sims = 0 # sum of similarities
    rating_diffs = 0 # rating differences, where similarity as a weight

    for b_idx in range(user_item_matrix.shape[0]): # go through all the other users with who a has similairities
        if b_idx == a_idx: # b is the target user
            continue 
            
        sim = similarity(a_idx, b_idx) # get the similarity of users
        if sim > 0:
            b_ratings = user_item_matrix[b_idx].toarray().flatten() # all the ratings of b
            b_rating_avg = b_ratings[b_ratings > 0].mean() if (len(b_ratings) > 0) else 0 # average rating of b, if no ratings this is 0
            b_movie_rating = b_ratings[movie_idx] # rating of b for the target movie
            if (b_movie_rating>0): # if b has rated the movie, update the sums
                rating_diffs += sim * (b_movie_rating - b_rating_avg)
                sum_sims += sim

    if sum_sims == 0: # no similarities
        return a_rating_avg # return the average rating of the target
    
    prediction = a_rating_avg + (rating_diffs/sum_sims)
    return prediction

In [9]:
# check that the prediction and similarity functions work
prediction(1,2)

4.309493046581272

(d) Design and implement a new similarity function for computing similarities between
users. Explain why this similarity function is useful for the collaborative filtering
approach. Hint: Exploiting ideas from related works are highly encouraged.

I decided to implement compute euclidean distance function and cosine distance function to measure similarity. It calculates the distance between two users. 
It is useful especially for the movies because it gives a distance between users, so if users have seen the same movies (distance is small) they have probably similar taste and it would be good to recommend other movies that the other uses has seen but the other one not.

In collaborative filtering in general, this is a useful because we assume that if two users liked similar items in the past, they are going to like similar items also in the future. So, if users A and B have similar preferences it estimates that B would prefer an item that is not rated by B but is rated well by A.
Especially in cases when users are not usually giving rates if they didn't like the item, this is useful. Or in cases when the estimated value is maybe something else than rates, like time spent in a site. Still there might be situations that the site is just forgotten to be open, but generally it is rare.

Other option is to compute the cosine distance which works better in item-to-item filtering. The smaller the return value is, the closer the items are each others. 

In [10]:
def euclideanDistanceUsers(a,b):
    a_ratings = user_item_matrix[a].toarray().flatten() # all the ratings of a
    b_ratings = user_item_matrix[b].toarray().flatten() # all the ratings of b
    common_ratings = (a_ratings > 0) & (b_ratings > 0) # common ratings

    if len(common_ratings) == 0: return 0 # no common ratings

    a_common_ratings = a_ratings[common_ratings] # ratings of a in common ratings
    b_common_ratings = b_ratings[common_ratings] # ratings of b in common ratings

    return spatial.distance.euclidean(a_common_ratings, b_common_ratings)

In [11]:
def cosineDistanceUsers(a,b):
    a_ratings = user_item_matrix[a].toarray().flatten() # all the ratings of a
    b_ratings = user_item_matrix[b].toarray().flatten() # all the ratings of b
    common_ratings = (a_ratings > 0) & (b_ratings > 0) # common ratings

    if len(common_ratings) == 0: return 0 # no common ratings

    a_common_ratings = a_ratings[common_ratings] # ratings of a in common ratings
    b_common_ratings = b_ratings[common_ratings] # ratings of b in common ratings

    return spatial.distance.cosine(a_common_ratings, b_common_ratings)

In [46]:
# check the distance functions working
print("Euclidean distance of users 1 and 5: ", euclideanDistanceUsers(1,5))
print("Cosine distance of users 1 and 5: ", cosineDistanceUsers(1,5))

Euclidean distance of users 1 and 5:  2.23606797749979
Cosine distance of users 1 and 5:  0.010050506338833531


(e) Use the user-based collaborative filtering approach to produce group
recommendations. Specifically, first compute the movies recommendations for each
user in the group, and then aggregate the lists of the individual users, to produce a
single list of movies for the group. You will implement two well established aggregation
methods for producing the group recommendations.
The first aggregation approach is the average method. The main idea behind this
approach is that all members are considered equals. So, the rating of an item for a
group of users will be given be averaging the scores of an item across all group
members. 

The second aggregation method is the least misery method, where one member can act
as a veto for the rest of the group. In this case, the rating of an item for a group of users
is computed as the minimum score assigned to that item in all group members
recommendations.

In [13]:
def similarityMatrix(matrix):
    dense_matrix = matrix.toarray()
    similarity_matrix = np.corrcoef(dense_matrix)
    return similarity_matrix

In [14]:
# similarities between users in one matrix
user_similarity_matrix = similarityMatrix(user_item_matrix)

In [15]:
def recommend(a, k=40, num_recs=10): # if not other parameters than user given, compute with 40 most similar user and return 10 recommendations
    if a not in user_map: return [] # first check that the user is valid

    recs = {} # recommendations
    a_idx = user_map[a]
    a_ratings = user_item_matrix.getrow(a_idx).toarray().flatten()
    similar_users = np.argsort(user_similarity_matrix[a_idx])[-(k+1):-1][::-1] # k most similar user to a (that are not a)

    for b_idx in similar_users:
        similarity = user_similarity_matrix[a_idx, b_idx] # similarity of a and b from the matrix
        if similarity <= 0:
            continue
        b_ratings = user_item_matrix.getrow(b_idx).toarray().flatten() # all the ratings of b
        unrated = np.where(a_ratings == 0)[0] # movies that a has not rated
        b_rating_avg = b_ratings[b_ratings > 0].mean() if np.any(b_ratings > 0) else 0 # average rating of b

        predictions = similarity * (b_ratings[unrated] - b_rating_avg) # predictions for unrated movies
        # if a has not rated a movie, predict the rate
        for idx, movie_idx in enumerate(unrated):
            movie = items[movie_idx]
            if movie not in recs:
                recs[movie] = a_ratings[a_ratings > 0].mean() + predictions[idx] # add to recommendations the prediction for movie 

    topRecs = sorted(recs.items(), key=lambda x: x[1], reverse=True)[:num_recs] # sort s.t. the highest predictions are first
    return topRecs


In [16]:
# check that recommend function is working
recommend(1)

[(1079, 4.883854329725796),
 (1199, 4.883854329725796),
 (1266, 4.883854329725796),
 (1288, 4.883854329725796),
 (1466, 4.883854329725796),
 (2599, 4.883854329725796),
 (2683, 4.883854329725796),
 (2712, 4.883854329725796),
 (3060, 4.883854329725796),
 (16, 4.883854329725796)]

In [17]:
def averageAggr(group, movie):
    sum = 0.0
    for user in group:
        idx = user_map[user] # index of user
        ratings = user_item_matrix.getrow(idx).toarray().flatten() # all the rating of user
        movie_idx = item_map[movie] # get the index of movie
        movie_rating = ratings[movie_idx] # rating for movie 

        if movie_rating > 0: # user has rated 
            sum += movie_rating
        else: # user has not rated
            sum +=  prediction(user,movie)
    return sum/len(group)

In [18]:
def leastMiseryAggr(group, movie):
    min = 5
    for user in group:
        idx = user_map[user] # index of user
        ratings = user_item_matrix.getrow(idx).toarray().flatten() # ratings of user
        movie_idx = item_map[movie] # get the index of movie
        movie_rating = ratings[movie_idx] # rating for movie 

        if movie_rating > 0: # user has rated 
            if movie_rating < min: min = movie_rating
        else: # user has not rated
            pred = prediction(user, movie)
            if pred < min: min = pred
    return min

(f) Define a way for counting the disagreements between the users in a group and
propose a method that takes disagreements into account when computing suggestions
for the group. Implement your method and explain why it is useful when producing
group recommendations. Prepare also a short presentation (about 5 slides) to show
how your method works.

The user satisfaction value tells how happy/satisfied the user is about the group recommendation. 

In [19]:
def groupRecommend(group, agg=1): # agg 1 is average aggregation and 2 is least misery aggregation 
    group_recs = {}
    for a in group: # go through all the members in the group
        recommendations = recommend(a) # get recommendations to the user
        for movie, pred in recommendations: # for all the recommended movies, get the aggregation
            if agg == 1: # use average aggregation
                group_recs[movie] = averageAggr(group,movie)
            elif agg == 2: # use least misery aggregation
                group_recs[movie] = leastMiseryAggr(group,movie)
            else: return [] # invalid method
    top_group_recs = sorted(group_recs.items(), key=lambda x: -x[1])[:5] # top five recommends for the group
    return top_group_recs

In [20]:
#check that group recommenders are working
group = {1,2,3,4,5}
# group_recommed(group)
recs = groupRecommend(group,2)


In [21]:
def userSatisfaction(user, group_recs):
    idx = user_map[user] # index of user
    ratings = user_item_matrix.getrow(idx).toarray().flatten() # all the rating of user
    
    # initialize sums
    predicted = 0 
    actual = 0
    for movie, pred in group_recs:
        movie_idx = item_map[movie] # get the index of movie
        movie_rating = ratings[movie_idx] # rating for movie 

        if movie_rating > 0: # user has rated
            actual += movie_rating
            predicted += pred # add the predicted value to sum of predictions
            
    if predicted == 0: return 0
    return actual/predicted

In [22]:
def groupSatisfaction(group, recs):
    sum = 0
    for user in group:
        sum += userSatisfaction(user, recs)
    return sum/len(group)

In [23]:
# check that group satisfaction is working
group = {1,2,3,4,5}
groupSatisfaction(group,recs)

0.8462663346778141

In [24]:
def disagreement(group,recs):
    user_sats = {}
    for user in group: 
        user_sats[user] = userSatisfaction(user, recs) # satisfaction for al the user in the group

    satisfaction_values = [sat for user, sat in user_sats.items()] # satisfaction values of the users in group
    max_satisfaction = max(satisfaction_values)
    min_satisfaction = min(satisfaction_values)

    return max_satisfaction - min_satisfaction

In [25]:
# check that disagreement works
disagreement(group,recs)

1.580763921685837

In [26]:
def groupRecommendWDisagreement(group, recs, agg=1):
    group_disagree = disagreement(group, recs) # disagreement of the group
    final_recs = {}
    for user in group:
        user_sat = userSatisfaction(user, recs) # users satisfaction
        weight = 1 / (1 + abs(user_sat - group_disagree)) # weight based to satisfaction
        user_recs = recommend(user)
        aggregated_score = 0
        # count new recommendations with weight
        for movie, pred in user_recs:
            if agg == 1: # use average aggregation
                aggregated_score = averageAggr(group, movie)
            elif agg == 2: # use least misery aggregation
                aggregated_score = leastMiseryAggr(group,movie)
            else: return [] # invalid method

            final_recs[movie] = aggregated_score * weight

    top_recs = sorted(final_recs.items(), key=lambda x: -x[1])[:5]
    return top_recs

In [27]:
# check that group recommend with disagreements is working
groupRecommendWDisagreement(group,recs)

[(1266, 4.141064483598427),
 (1288, 4.0838494917596115),
 (16, 4.007079640168898),
 (2599, 3.987227620294995),
 (3060, 3.9396436401196104)]

In [28]:
# compare the previous results to the ones without disagreements
groupRecommend(group)

[(1089, 4.156517878988662),
 (50, 4.152384311253367),
 (1266, 4.141064483598427),
 (1213, 4.140141330443503),
 (593, 4.111418556596455)]

PART II

Motivated by the sequential methods we discussed in class, our goal here is to design (7
points) 
and implement (7 points) a new method for producing sequential group
recommendations. 

Also, provide detailed explanations and clarifications about why the
method you propose works well for the case of sequential group recommendations (6
points). 

Prepare also a short presentation (about 5 slides) to show how your method
works (5 points).

DESIGN:
I am going to use the hybrid aggregation which is a weighted combination of the average and least misery aggregations. The methods of aggregations are implemented above in the PART I (and I have modified them to use the sparse user-item matrx instead of dataframe).

Sequental group recommendations take into account also the history. So if an user was not satisfied in the past, it tries to make sure that the user is going to be satisfied in the future. I am going to add the history to the algorithm as a parameter.

For computing disagreement I am using the previous implemented function disagreement (which is calling also userSatisfaction) above. I am also using recommend function that is defined above. This uses the average aggregation as a default but can be easily switched to use the least misery aggregation by adding parameter 2 to the input.

I am going to use "Algorithm 1: Sequential Group Recommendation Algortihm" from the article "Fair Sequential Group Recommendations", written by M. Starigi, E. Pitoura, J. Nummmenmaa and K.Stefanidis as a base of my sequential group recommender algorithm. Compared to the original algorithm I am adding a history parameter to convey the history information to the following iteration to make possible to check that the item is not already recommended, such that the algortihm is not recommending something it has already recommended in the previous iterations. 

Besides the history, the algorithm takes also as an input a group (group), an iteration number (iter) and size of the recommendation list (k), which I am setting 5 as a default. First the algorithms computes a disagreement value if the iteration is bigger than 1. In the first iteration this value is 0, because there is no previous iterations (history) according to what the disagreement could be calculated. In that case the weight is the same for all the users. Then the algortihm gets preferences of every user in the group and goes through all the preferences and items on those preference lists and computes to it the weighted score based on the hybrid aggregation. The score will be added to the recomedations list.

After the algortihm remove from recommendations all the items that are already recommended in the previous iterations (all the items in the history) to not recommend the same items over and over again. The recommendations are sorted and added to the history before returning both the top recommendations and the history. 

In my opinion it is simpler to return also the top recommendations of the iteration so that the user can easily see them if needed, without going through the history.

In [29]:
def sequentialGroupRecommend(group, history, iter, k=5): # history means previous recommendations

    a = disagreement(group, history[iter-2]) if iter > 1 else 0 # if not the first iteration, compute the disagreement factor

    users_preferences = {user: recommend(user) for user in group} # single user preferences 
    recs = {}

    for user, user_recs in users_preferences.items():
        for movie,pred in user_recs:
            score = (1-a) * averageAggr(group, movie) + a * leastMiseryAggr(group,movie) # weighted score

            if movie not in recs:
                recs[movie] = 0
            recs[movie] += score # add the score to the corresponding movie
    
    previous_recs = {movie for recs in history for movie, _ in recs}
    recs = {movie: score for movie, score in recs.items() if movie not in previous_recs} # do not recommend a movie if already recommended

    top_recs = sorted(recs.items(), key=lambda x: -x[1])[:k]
    history.append(top_recs)
    return top_recs, history


In [30]:
# check the sequential group recommendation method is working with the movie data
group = {1,2,3,4,5}
iter_1, history_1 = sequentialGroupRecommend(group, [], 1)
print("Iteration 1 Recommendations:")
print(iter_1)
iter_2, history_2 = sequentialGroupRecommend(group, history_1, 2)
print("Iteration 2 Recommendations:")
print(iter_2)
iter_3, history_3 = sequentialGroupRecommend(group, history_2, 3)
print("Iteration 3 Recommendations:")
print(iter_3)

Iteration 1 Recommendations:
[(1089, 12.469553636965987), (356, 7.839970240109362), (110, 7.666169740701039), (50, 4.152384311253367), (1266, 4.141064483598427)]
Iteration 2 Recommendations:
[(1208, 3.0533174761844926), (1213, 3.012740962853127), (593, 2.9959005146513706), (7153, 2.978192608327186), (4993, 2.9728842905092807)]
Iteration 3 Recommendations:
[(1198, 2.432522531131471), (1288, 2.3505189013823857), (2028, 2.341046512902918), (223, 2.3066562013081873), (8368, 2.3002375619842432)]


This method works well because it takes into account the history of users satisfaction and recommendations such that if a user was not satisfied in the previous iteration, it will have a bigger weight when making a new recommendations. The sequential hybrid model I have used quarantees that both, the group satisfaction and variance between users satisfaction scores are considered because it is a combination of the average and least misery aggregation models. 

In the first iteration the weight is same for all of the users. Practically this means that the average aggregation is used at this point. In the following iterations a disagreement value (a) makes sure that the recommendations are more sensitive to variations in satisfactions within the group. This enolve the recommendations and balance between group satisfaction and variance between users satisfaction.

Also, this method makes sure that the list of the recommendations is not containing the same items but it is proposing everytime new items. This is done by filtering out all the items that are already in the history.

The method supports different ways of aggregating user preferences to form group recommendations. As a default it uses the average aggregation  (agg = 1): In this method, the recommendation score for each movie is computed by averaging the individual ratings of all group members. This approach works well when users' preferences are similar, as it captures a middle-ground approach where the group is happy with a consensus recommendation. On the other hand, it is possible also use the least misery aggregation (agg = 2):
This method is designed to minimize the dissatisfaction for the group members, It works well when there is significant disagreement between users, as it ensures that the leas liked recommendation is minimized. This aggregation method works well when users have highly divergent preferences, as it reduces the chance of recommending a movie that would significantly lower group satisfaction.

Overall, the method is good for sequential group recommendations because it balances the need for both personalization and group satisfaction. The iterative process helps the method improve over time, creating more accurate and engaging recommendations for the group.

PART III

In Part III of the project, we would like to promote diversity in recommender systems to
offer a wide range of content and ensure that this content is relevant and engaging for all members of the group. 

Motivated by the diversification methods we discussed inclass, 
design (7 points) and implement (7 points) a new method for producing diverse sequential group recommendations. 

Explain why this method works well for the case of sequential group recommendations (6 points). 

Prepare also a short presentation (about 5 slides) to show how your method works (5 points).

The goal of the method is to provide diverse group recommendations  sequentially and mantain the quality of the recommendations. To avoid redundancy in the recommendations, every iteration provides unique suggestions to the group based on their preferencies and the history.

Input parameters of the method are:
- group: the set of users, the recommendations are given
- history: a list of the movies already recommended in the past iterations
- iter: the iteration number, used to adjust the diversity radius
- radius: a diversity factor, controls how spread the recommendations will be
- k: the number of recommendations from each iteration

The point of adjusted radius is to increase the diversity of recommendations with each  iteration to get more variety and to avoid repetitive recommendations.

To take into account individual preferences the method generate first all individual recommends for all the group members.

To ensure fresh recommedations, the previously recommended movies (movies in the history) will be filtered out from the new recommendations.

For generating recommendations from the list of all individual recommendations is used Greedy DisC Algorithm which returns a diverse subset of movies. It maximize diversity by selecting movies as dissimilar as possible within the radius.

After generating the recommendations, they are added to the history for the future iterations.

The method returns the top k diverse movies of the iteration and the history.

Before implementing the method for diverse sequential group recommendations, I need to implement some other algorithms that are used by the method.

First, I implement algorithm to compute teh Euclidean distance for to items.

The Euclidean distance is a measure of similarity or dissimilarity between two items. Smaller distance indicates more similarity, in this case, the movies have received similar ratings from users.

This algortihm takes to movies as an input, it checks if they have any common ratings, meaning the same user have rated both of the movies, and if no the distance is returned as 0. Otherwise it computes the euclidean distance between the common ratings.

In [31]:
def euclideanDistanceItems(item1, item2):
    idx1 = item_map[item1] # index of item1
    idx2 = item_map[item2] # index of item2
    item1_ratings = user_item_matrix.getcol(idx1).toarray().flatten() # ratings of the item1
    item2_ratings = user_item_matrix.getcol(idx2).toarray().flatten() # ratings of the item2
    common_ratings = (item1_ratings > 0) & (item2_ratings > 0) # common ratings

    if len(common_ratings) == 0: return 0 # no common ratings

    item1_common_ratings = item1_ratings[common_ratings] # ratings of a in common ratings
    item2_common_ratings = item2_ratings[common_ratings] # ratings of b in common ratings

    return spatial.distance.euclidean(item1_common_ratings, item2_common_ratings)

In [32]:
# check that euclidean distance for items is working
euclideanDistanceItems(1,5)

8.817596044274199

Second, I need an algorithm to get the neighbors of an item. Other words, this algorithm is returning all the movies that are among r distance from a given movie.

It takes as an input the target movie and the maximum distance r. Then it goes through all the movies and returns a list of movies that have as a maximum r distance from the target movie. the distance is computed as Euclidean distance.

In [33]:
def neighbors(item, r):
    neighbor_list = []
    for other_item in items:
        if other_item != item:  # not the item itself
            if euclideanDistanceItems(item, other_item) <= r:
                neighbor_list.append(other_item)
    return neighbor_list

In [34]:
# Check that the neighbors algorithm is working
neighbors(1,3)

[804,
 940,
 943,
 1024,
 1226,
 1473,
 1793,
 1804,
 1927,
 2033,
 2048,
 2093,
 2099,
 2366,
 2492,
 2596,
 2644,
 2648,
 2654,
 2899,
 3053,
 3062,
 3729,
 4006,
 86345,
 89774,
 112552,
 114060,
 131724,
 1124,
 2851,
 3024,
 4518,
 5181,
 5746,
 5764,
 5919,
 6835,
 7899,
 7991,
 106,
 171,
 176,
 190,
 450,
 492,
 538,
 599,
 759,
 937,
 1046,
 1103,
 1279,
 1733,
 1734,
 1834,
 1860,
 1966,
 2203,
 2204,
 2282,
 2351,
 2390,
 2583,
 2843,
 2874,
 2921,
 3079,
 3204,
 3365,
 3508,
 3538,
 3788,
 3851,
 4020,
 4021,
 4029,
 4033,
 4074,
 4121,
 4166,
 4252,
 4260,
 4273,
 4347,
 4381,
 4765,
 4902,
 290,
 531,
 8,
 26,
 41,
 43,
 61,
 86,
 100,
 113,
 146,
 177,
 179,
 189,
 201,
 205,
 209,
 210,
 212,
 219,
 230,
 240,
 243,
 251,
 254,
 262,
 270,
 271,
 279,
 281,
 289,
 291,
 302,
 304,
 310,
 313,
 336,
 340,
 343,
 347,
 352,
 354,
 358,
 359,
 371,
 378,
 381,
 416,
 426,
 458,
 460,
 468,
 469,
 477,
 493,
 505,
 510,
 536,
 548,
 606,
 631,
 662,
 694,
 697,
 747,
 775,


Third, I need a Greedy-DisC algorithm to select a diverse subset of items from a set of recommendations. (DisC means dissimilar and covering.) The goal is to maximize the overall satisfaction and ensure the diversity by selecting as dissimilar items as possible. 

To achieve the goal the algorithm uses a greedy approach and a diveristy measure (Euclidean distance). It chooses the highest-rated movie from the available (white) movies.

The algortith takes the list of all individual users recommendations, radius (the diversity threshold, maximum distance) and k (number of items to select) as input parameters. It first combines the recommedations from all users and keeps the highest score if the movie is recommended by several users. The combined recommendations are sorted in descending order, so that highest rated are the first ones.

Initially, all the movies are colored white. Then the algorithm works iteratively choosing the highest scored white movie. The selected movie is colored black and all the movies the given radius distance from it are colored gray. This is repeated until there is no white movies left or until the size of the subset is k.

In [35]:
def GreedyDisC (all_recs, radius = 5, k=5):

    combined_recs = {} # combined recommendations from all the users in the group
    for user_recs in all_recs:
        for movie, score in user_recs:
            if movie not in combined_recs:  # the highest score if the movie is recommended by many users
                combined_recs[movie] = score
            else:
                combined_recs[movie] = max(combined_recs[movie], score)


    sorted_recs = sorted(combined_recs.items(), key=lambda x: x[1], reverse=True) # combined recommendations sorted by scores, descending
    
    S = [] # selected set
    colored_items = {movie: "w" for movie, _ in sorted_recs}  # map to track the colors

    while len(S) < k and any(color == "w" for color in colored_items.values()): # while the size of S is smaller than k and there is still white values
        for movie, _ in sorted_recs: 
            if colored_items[movie] == "w": 
                selected_movie = movie # the first white movie
                break

        S.append(selected_movie) # add selected movie to S
        colored_items[selected_movie] = "b"  # color selected movie black

        for other_movie, _ in sorted_recs:
            if colored_items[other_movie] == "w" and euclideanDistanceItems(selected_movie, other_movie) <= radius:
                colored_items[other_movie] = "g" # color gray all the white neighbours of selected movie among r distance

    return S


I implemented also a method to generate group recommendations using the Greedy-DisC algortihm to check it is working before moving forward.

This method gives normal group recommendations (non sequential).

In [36]:
def disCDiverseGroupRecs(group):
    all_recs = [] # all individual recommendations of the group
    for a in group: # go through all the members in the group
        a_recs = recommend(a) # get recommendations to the user
        all_recs.append(a_recs) # add to all recs

    S = GreedyDisC(all_recs) # compute a DisC diverse subset 
    return S

In [37]:
# check that disC diverse group recommendations (greedy disC algorithm) work
group = {1,2,3,4,5}
disCDiverseGroupRecs(group)

[1079, 1288, 2599, 2683, 356]

Finally, I implemented the method to generate sequential group recommendations as explained before.

In [38]:
def disCDiverseSequentialGroupRec(group, history, iter, radius=3, k=5):

    radius = radius + 0.5 * (iter - 1) # increase radius with each iteration

    all_recs = [] # all individual recommendations of the group
    for a in group: # go through all the members in the group
        a_recs = recommend(a) # get recommendations to the user
        all_recs.append(a_recs) # add to all recs

    history_set = set(history) # history as a set, s.t. it is easier to handle
    filtered_recs = []
    for user_recs in all_recs:
        filtered_recs.append([(movie, score) for movie, score in user_recs if movie not in history_set]) # filter recs that are already in the history

    diverse_recs = GreedyDisC(filtered_recs, radius=radius, k=k) # get the DisC diverse subset

    history.extend(diverse_recs) # add the new recs to the history

    return diverse_recs, history # return both the diverse recommendations and the history

In [39]:
# check that disC diverse sequential group recpmmendations work
group = {1, 2, 3, 4, 5}
history_0 = []

# Perform sequential recommendations
iter_1, history_1 = disCDiverseSequentialGroupRec(group, history_0, iter=1, radius=3, k=5)
print("Iteration 1 Recommendations:", iter_1)

iter_2, history_2 = disCDiverseSequentialGroupRec(group, history_1, iter=2, radius=3, k=5)
print("Iteration 2 Recommendations:", iter_2)

iter_3, history_3 = disCDiverseSequentialGroupRec(group, history_2, iter=3, radius=3, k=5)
print("Iteration 3 Recommendations:", iter_3)

Iteration 1 Recommendations: [1079, 1199, 1266, 1288, 1466]
Iteration 2 Recommendations: [2599, 2683, 2712, 16, 356]
Iteration 3 Recommendations: [3060, 1097, 539, 47, 296]


This method works well for sequential group recommends because it balances the trade-off between individual user preferences and group diversity. 

The Greedy-DisC algortihm ensures that the items with highest ratings among all the group members are selected. The algorithm maximizes the group satisfaction because it increases the likelyhood that a user will find something attractive in the set of recommendations. It also takes into account the history of the recommendations which ensure fresh recommendations.

To avoid repetitive recommendations, the algorithm uses Euclidean distance to make sure the recommendations are diverse. In a group, users may have different preferences, and that's why especially for groups, it is important to generate diverse recmmendations.

The diversity threshold, radius, determines how different the items must be. Too similar items (the items that have smaller distance than radius from the target item) are excluded to make the final recommendations contain a variety of choices.

The increasing radius ensures greater diversity in later iterations, which help the algorithm not to provide similar recommendations repeately but keep the recommendations fresh and diverse. Also, using of history and filtering already previously recommended movies out, ensures not repetitive recommendations.

In conclusion, the method works well for sequential group recommendations because it maximizes the group satisfaction, ensures diversity, adapts to sequential interactions, handles the history of recommendations and uses a greedy approach. 



PART IV

In this part of the project, the goal is to produce counterfactual explanations for a group
of users. 

Specifically, design (10 points) and implement (10 points) a method that
generates counterfactual explanations that adhere to some characteristics to be more
in tune with group recommendations. 

For example, an explanation that only consists of items interacted by a single user is undesirable since it would single out that user to the rest of the group. Ideally, we would like explanations that consist of items most users
have interacted with to make the explanation fairer. In this context, it means that no user should have changed their preferences for the group, but the group as a whole is responsible for the suggestions provided by the system. 

Prepare also a short presentation (about 5 slides) to show how your method works (5 points).

MY METHOD:

First I am going to use the previously implemented method to get the recommendation list for the group.

To ensure fairness of counterfactual explanations I create a set of items that contains the items all the group members have rated. This way no user is singled out.

After I remove one of these items per time and see if it affects to the original recommendations. If an item is not anymore in the recommendations, it means that the removed item explains the recommendation of the item. So the removed item is the explanation.

The define the search area, I need a method to find the common ratings of the group members. This method takes a group as an input and gets the ratings of all the group members. The first users ratings are used to initialize the set of common ratings and for all the other group members is taken an intersect of the common ratings and their individual ratings. The method returns the set of common ratings for all the users in the group.

To get the first recommendations for the group I am using the groupRecommend -method from part I. Before implementing the counterfactual explanation method, I have to also implement an method that is creating group recommendations from a set of items. For that I am implementing a new method groupRecommendUsingItemSet. This method is similar to the groupRecommend -method, but it does not get the individual recommendations to which to get the aggregation, but it takes the set as an input. Basically, I could create the same group recommendations with both methods, if I just generate a set of individual recommendations before and give is as an input to the new method. 

This method uses the average aggregation. It takes the group, the item set and the size of top recommendations to return with a default value as an input. Then it gets the average aggregation for all the items in the set and in the end sorts them in descending order and returns the top k of them.

Finally, I can implement the counterfactual Explanations. Because the point is that systematically remove some items from the user's previous interactions and then call on the recommendation system again to check if the item they want to provide an explanation for is removed from the user's suggestions and for the group to ensure the fairness to remove items from their common ratings and see if it affects to the group recommendations.

This method takes a group as an input and generates group recommendations and common ratings for the group, as explained before. Then for every movie in the common ratings it generates a new group recommendations with groupRecommendUsingItemSet -method giving the common ratings without the one movie as an input set. After it compares the original group recommendations with this new recommendations and see if there is all the same movies or if some are missing. If a movie that is in the original recommendations is missing from the new recommendations, we can say that the removed movie is the explanation for the missing movie. In the end, the method returns the explanation, which is the set of tuples of the movies (to explain) and the explanation movies.

NOTE ABOUT THE SEARCH AREA: 

When testing my original findCommonRatings method, that was finding the set of movies all the group members had rated, with different inputs I realized that the set of common ratings is often pretty small or empty. To provide meaningful counterfactual explanations in this case, I decided to create a new method, that is finding the common ratings for a group such that the items are rated by at least the given percentage of the members.

This method takes the group and the percentage with a default value as an input. So the fairness is adjustable and the user can decide to how big part of the group the movie has to be rated by. The method go through all the users in the group and combines their individual ratings to a set of tuples of movies and their scores such that the score is the maximum score the movie has got among the users. It also maintain another tuple of movies and their count, meaning how many times the movie occurs in the combined ratings. After it checks if the movie count is at least the size of the group times the given percentage, and if that is true, the movie and its highest score is added to common ratings.

This way, giving 1 as a percentage in input, the method return the set of movies that are rated by all the members in the group. If the percentage is 0.5 it returns the movies that are rated by at least half of the group, and so on.

In [40]:
def findCommonRatingsForGroup(group,p=1): # p=1 means 100% of the group has rated the movie, p=0.5 means half of the group, and so on
    # check p is valid; if not, set p=1
    if (p<0.0) | (p>1.0): p=1

    combined_ratings = {}
    common_ratings = {}
    movie_count = {}
    min_count = len(group) * p # in count is the size times the percentage 

    for user in group:
        idx = user_map[user]
        user_ratings = user_item_matrix.getrow(idx).toarray().flatten() # ratings of the user
        for movie_idx, score in enumerate(user_ratings): # movie_idx is the position of the score in the ratings
            if score > 0:  # only rated movies
                if movie_idx not in combined_ratings: # the first occurence of the movie
                    combined_ratings[movie_idx] = score # add to combined ratings
                    movie_count[movie_idx] = 1 # the first occurence, so the count is 1
                else:
                    combined_ratings[movie_idx] = max(combined_ratings[movie_idx], score) # the max score if the movie is rated by many users
                    movie_count[movie_idx] += 1 # increase the count
    
    for movie_idx, count in movie_count.items():
        if count >= min_count:
            common_ratings[movie_idx] = combined_ratings[movie_idx] # if the movie(index) has occured at least the min_count times in the combined ratings, add to common ratings

    reverse_item_map = {v: k for k, v in item_map.items()} # because we just worked with movie indices, not with the real movie ids, we need a reverse map to get the ids
    common_ratings_with_ids = {reverse_item_map[movie_idx]: score for movie_idx, score in common_ratings.items()} # common ratings with the movie id (not index)
        
    sorted_common_ratings = sorted(common_ratings_with_ids.items(), key=lambda x: x[1], reverse=True)
    return sorted_common_ratings # return movies and the highest scores of the common ratings

In [41]:
def groupRecommendUsingItemSet(group, items, k=10):
    group_recs = {}
    for movie, score in items: # for all the movies in the given set
        group_recs[movie] =  averageAggr(group,movie) # get the average aggregation

    top_group_recs = sorted(group_recs.items(), key=lambda x: -x[1])[:k] # top k recommendations for the group
    return top_group_recs

In [42]:
# check that two methods above work
group = {1,2,3,4,5}
common_rats = findCommonRatingsForGroup(group)
print("common ratings for group: ", common_rats)
group_recs = groupRecommendUsingItemSet(group,common_rats)
print("recommendations with the set of common items", group_recs)

common ratings for group:  []
recommendations with the set of common items []


I decided also add the precentage to the counterfactual explanation method, because it is easy to define it in the call of this method.

In [43]:
def counterfactualExp (group, p=1):
    common_ratings = findCommonRatingsForGroup(group, p)
    group_recs = groupRecommend(group)
    explanation = {}
    for movie_a, score_a in common_ratings:
        recs_without_movie = [(movie, score) for movie, score in common_ratings if movie != movie_a] # copy common recs without the movie to not change permanently the original set
        group_recs_without_movie = groupRecommendUsingItemSet(group, recs_without_movie) # get group recommendations without the movie
        
        movies_without_movie = [movie for movie, score in group_recs_without_movie] # just the movies from the recommendations, because no need for the score

        for movie_b, score_b in group_recs:
            if movie_b not in movies_without_movie: # if movie b is not in the recs anymore after removing movie a
                if movie_b not in explanation:
                    explanation[movie_b] = []  # initialize as an empty list if not already present
                explanation[movie_b].append(movie_a) # movie a is an explanation for movie b to be recommended
    return explanation

In [44]:
# check the working of the counterfactual explanation method.
group = {1,2,3,4,5}
explanation = counterfactualExp(group,0.6)
for a,b in explanation.items():
    print("If you had not liked the item(s) ", b, " then item ", a, " would not have been suggested.")

If you had not liked the item(s)  [296, 457, 527, 608]  then item  1089  would not have been suggested.
If you had not liked the item(s)  [296, 457, 527, 608]  then item  50  would not have been suggested.
If you had not liked the item(s)  [296, 457, 527, 608]  then item  1266  would not have been suggested.
If you had not liked the item(s)  [296, 457, 527, 608]  then item  1213  would not have been suggested.
If you had not liked the item(s)  [296, 457, 527, 608]  then item  593  would not have been suggested.


This is working well because the available searching area is the items that the group members have interacted with. The percentage how many group members have interacted, is adjustable; it can be defined as 0.0 when it enough that just 1 member has rated the movie or 1.0 when all the members have rated the movie in the searching area, or anything in the middle. Adjusting this percentage it is possible to affect also to the fairness of the explanations.

The method sequentally test removing one item from the searching area, if it affects to the recommendations for the group. 

For the future, something to think and improve is to take user intesity into account to ensure even more fairness. If the searching area is now all the movies 60% of the group has rated, it is possible that all the ratings are from the same members and one user is sindgled out.